# WP2 — Preprocessing starter (Student 1)

**What this notebook does.** Demonstrates the WP2 pipeline pattern against the synthetic bundle: reads the synthetic `covariates.parquet`, emits the four preprocessing artefacts specified in `docs/pipeline_contract_v1.md` §3 (cohort, valid_nights, missingness_report, loso_folds), and writes them to `synthetic/v1/preprocessing/`.

**Real-data mode.** When mcPHASES raw CSVs are available in `dataset/` (and converted via `scripts/convert_raw_to_parquet.py`), the notebook uses them instead of the synthetic fallback. Switch by setting `REAL_DATA = True` in the setup cell.

**Outputs.**
- `preprocessing/cohort.parquet`
- `preprocessing/valid_nights.parquet`
- `preprocessing/missingness_report.parquet`
- `preprocessing/loso_folds.parquet`


In [ ]:
import sys, os, subprocess
from pathlib import Path

REPO_ROOT = Path().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import numpy as np
import pandas as pd
from utils.preview import peek, summary

try:
    from google.colab import data_table
    data_table.enable_dataframe_formatter()
except ImportError:
    pass

DATA = Path('synthetic/v1')
DATA.mkdir(parents=True, exist_ok=True)
(DATA / 'preprocessing').mkdir(exist_ok=True)
(DATA / 'endpoint').mkdir(exist_ok=True)
(DATA / 'decisions').mkdir(exist_ok=True)
(DATA / 'evaluation').mkdir(exist_ok=True)
print(f'Pipeline root: {DATA}')


## 1. Choose data source


In [ ]:
REAL_DATA = False  # Flip to True once mcPHASES raw files + parquet conversion are in place.

if REAL_DATA and (Path('dataset_parquet/heart_rate.parquet').exists()):
    print('Using real mcPHASES data.')
    source = 'real'
else:
    print('Using synthetic bundle (synthetic/v1/).')
    source = 'synthetic'


## 2. Build cohort table

For the synthetic bundle: every participant is included; cohort fields are re-derived from `covariates.parquet`. For real data: apply the WP2 inclusion/exclusion rule (to be frozen by Student 1). The schema is the same either way.


In [ ]:
if source == 'synthetic':
    cov = pd.read_parquet(DATA / 'covariates.parquet')
    prob = pd.read_parquet(DATA / 'probability_table.parquet')
    n_nights = prob.groupby('participant_id').size().rename('r1_valid_nights').reset_index()
    # Synthetic has no true R1/R2 split — pretend everyone has R1, R2 iff has_round_2.
    cohort = cov[['participant_id', 'has_round_2']].copy()
    cohort['included'] = True
    cohort['exclusion_reason'] = pd.Series([None] * len(cohort), dtype='string')
    cohort['has_round_1'] = True
    cohort = cohort.merge(n_nights, on='participant_id')
    cohort['r2_valid_nights'] = np.where(cohort['has_round_2'], cohort['r1_valid_nights'] // 3, 0)
    cohort['r1_total_days'] = (cohort['r1_valid_nights'] * 1.1).astype(int)
    cohort['r2_total_days'] = np.where(cohort['has_round_2'], (cohort['r2_valid_nights'] * 1.1).astype(int), 0)
    cohort = cohort.astype({
        'participant_id': 'string', 'included': 'bool',
        'has_round_1': 'bool', 'has_round_2': 'bool',
        'r1_total_days': 'int32', 'r2_total_days': 'int32',
        'r1_valid_nights': 'int32', 'r2_valid_nights': 'int32',
    })
else:
    raise NotImplementedError('Real-data cohort build — TODO for WP2 Milestone 1.')

cohort.to_parquet(DATA / 'preprocessing/cohort.parquet', index=False)
peek(DATA / 'preprocessing/cohort.parquet')


## 3. Build valid_nights table

One row per participant × study-day. In the synthetic case, we materialise one valid night per night_index the generator produced.


In [ ]:
if source == 'synthetic':
    vn = prob[['participant_id', 'night_index']].copy()
    vn['study_interval'] = 2022
    vn['day_in_study'] = vn['night_index']
    vn['is_valid_night'] = True
    vn['exclusion_reason'] = pd.Series([None] * len(vn), dtype='string')
    vn['quality_flag'] = pd.Series(['ok'] * len(vn), dtype='string')
    vn = vn[['participant_id', 'study_interval', 'day_in_study', 'is_valid_night',
             'exclusion_reason', 'night_index', 'quality_flag']]
    vn = vn.astype({
        'participant_id': 'string', 'study_interval': 'int16', 'day_in_study': 'int32',
        'is_valid_night': 'bool', 'night_index': 'Int32', 'quality_flag': 'string',
    })
else:
    raise NotImplementedError('Real-data valid-night filter — TODO for WP2 Milestone 2.')

vn.to_parquet(DATA / 'preprocessing/valid_nights.parquet', index=False)
peek(DATA / 'preprocessing/valid_nights.parquet')


## 4. Build missingness report

Per-modality coverage. Synthetic values derived from modality flags in `covariates.parquet`.


In [ ]:
if source == 'synthetic':
    rows = []
    for _, r in cov.iterrows():
        for modality, flag_col in [('temperature', 'has_temperature'), ('hr', 'has_hr_hrv'),
                                    ('hrv', 'has_hr_hrv'), ('glucose', 'has_cgm'),
                                    ('self_report', 'has_self_report'),
                                    ('hormones_lh', 'has_round_2'),
                                    ('hormones_e3g', 'has_round_2'),
                                    ('hormones_pdg', 'has_round_2')]:
            rows.append({
                'participant_id': r['participant_id'],
                'modality': modality,
                'days_total': 30,
                'days_available': 30 if r[flag_col] else 0,
                'coverage_pct': 1.0 if r[flag_col] else 0.0,
            })
    miss = pd.DataFrame(rows).astype({
        'participant_id': 'string', 'modality': 'string',
        'days_total': 'int32', 'days_available': 'int32', 'coverage_pct': 'float32',
    })
else:
    raise NotImplementedError('Real-data missingness — TODO for WP2 Milestone 3.')

miss.to_parquet(DATA / 'preprocessing/missingness_report.parquet', index=False)
peek(DATA / 'preprocessing/missingness_report.parquet')


## 5. Build LOSO fold assignment

Primary LOSO uses `participant_id` directly. This file is the 5-fold grouped-CV fallback per plan §3.6.


In [ ]:
import hashlib
def fold_of(pid: str) -> int:
    return int.from_bytes(hashlib.md5(pid.encode()).digest()[:4], 'big') % 5

folds = pd.DataFrame({
    'participant_id': cohort['participant_id'],
    'fold_id_5fold': [fold_of(p) for p in cohort['participant_id']],
}).astype({'participant_id': 'string', 'fold_id_5fold': 'int8'})

folds.to_parquet(DATA / 'preprocessing/loso_folds.parquet', index=False)
peek(DATA / 'preprocessing/loso_folds.parquet')


## 6. Sanity checks


In [ ]:
assert len(cohort) == 42, f'expected 42 participants, got {len(cohort)}'
assert set(cohort['participant_id']) == set(cov['participant_id']), 'cohort participant set mismatch'
assert len(vn) == len(prob), 'valid_nights row count should equal probability_table row count on synthetic'
assert folds['fold_id_5fold'].between(0, 4).all(), 'fold_id out of range'
print('WP2 preprocessing outputs: OK')


## Next

Student 1: adapt this notebook for the real mcPHASES data. The operational rules that need to be frozen (tracked in `docs/pipeline_contract_v1.md` §11):
- Valid-night rule (plan §4.3 proposes ≥4h continuous 22:00–06:00).
- Quality-flag taxonomy (`"ok"` / `"low_signal"` / `"dropout"`).
- Inclusion/exclusion criteria for the cohort.

After real-data WP2 outputs land, the endpoint (WP3, notebook 02) reads from `preprocessing/valid_nights.parquet` as the source of truth for which nights exist.
